# Collocation

In [1]:
import polars as pl
import polars_corpus as plc
import math

In [2]:
bnc = pl.read_parquet("bnc.parquet")

In [3]:
m = plc.search_cqp(bnc, '[token="dangerous"]')

In [4]:
collocs = m.collocates("token", window=3).sort(
    by=pl.col("freqs").struct.field("f12"), descending=True
)
collocs

collocate,freqs
str,struct[4]
""".""","{1760,32676,4715138,112429158}"
""",""","{1501,32676,5017057,112429158}"
"""and""","{1230,32676,2506072,112429158}"
"""a""","{1115,32676,2036673,112429158}"
"""the""","{1088,32676,5405646,112429158}"
…,…
"""unacceptable""","{5,32676,1206,112429158}"
"""stay""","{5,32676,11397,112429158}"
"""nuclear""","{5,32676,6975,112429158}"


In [5]:
ll = (
    collocs.with_columns(LL=pl.col("freqs").corpus.loglik())
    .sort(by="LL", descending=True)
    .head(20)
)
ll

collocate,freqs,LL
str,struct[4],f64
"""the""","{1088,32676,5405646,112429158}",174.157016
"""of""","{604,32676,3021526,112429158}",98.606426
"""I""","{118,32676,861655,112429158}",87.830013
""")""","{37,32676,397970,112429158}",73.191158
"""her""","{26,32676,287910,112429158}",54.686633
…,…,…
"""has""","{34,32676,254068,112429158}",26.999713
"""had""","{70,32676,417684,112429158}",25.799102
"""did""","{11,32676,126086,112429158}",24.841214


In [7]:
# double checking LL for 'potentially dangerous'

# a = frequency of node-collocate pairs
a = 154
# b = frequency of node without collocate
b = 32676 - a
# c = frequency of collocate without node
c = 2373 - a

# d = words in corpus - occurrences of node and collocate
N = 112429158
d = N - a - b - c

LL2 = 2 * (
    a * math.log(a)
    + b * math.log(b)
    + c * math.log(c)
    + d * math.log(d)
    - (a + b) * math.log(a + b)
    - (a + c) * math.log(a + c)
    - (b + d) * math.log(b + d)
    - (c + d) * math.log(c + d)
    + (N) * math.log(N)
)
LL2

1370.0401501655579